In [ ]:
import os
import pickle
import jsonlines
import pandas as pd
import numpy as np
import json
import copy
from tqdm import tqdm

In [ ]:
prompt_template = "The point of interest has following attributes: \n name is <NAME>; category is <CATEGORY>; type is <TYPE>; open status is <OPEN>; review count is <COUNT>; city is <CITY>; average score is <STARS>."

In [ ]:
data = json.load(open("./handled/item2attributes.json", "r"))

In [ ]:
item_data = {}
for key, value in tqdm(data.items()):
    item_str = copy.deepcopy(prompt_template)
    item_str = item_str.replace("<NAME>", value["name"])
    cate_str = ""
    for cate in value["categories"]:
        cate_str += (cate + " ")
    item_str = item_str.replace("<CATEGORY>", cate_str)
    item_str = item_str.replace("<TYPE>", value["type"])
    item_str = item_str.replace("<OPEN>", str(value["open"]))
    item_str = item_str.replace("<COUNT>", str(value["review_count"]))
    item_str = item_str.replace("<CITY>", value["city"])
    item_str = item_str.replace("<STARS>", str(value["stars"]))
    
    item_data[key] = item_str

In [ ]:
json.dump(item_data, open("./handled/item_str.json", "w"))

In [ ]:
item_data = json.load(open("./handled/item_str.json", "r"))

In [ ]:
import jsonlines

def save_data(data_path, data):
    '''write all_data list to a new jsonl'''
    with jsonlines.open("./handled/"+ data_path, "w") as w:
        for meta_data in data:
            w.write(meta_data)

id_map = json.load(open("./handled/id_map.json", "r"))["item2id"]
json_data = []
for key, value in item_data.items():
    json_data.append({"input": value, "target": "", "item": key, "item_id": id_map[key]})

save_data("item_str.jsonline", json_data)

In [ ]:
# Configure the embedding endpoint without storing credentials in this notebook.
# Set these environment variables before starting Jupyter.
import os
import requests
import time

EMBEDDING_URL = os.environ.get("CSBREC_EMBEDDING_URL", "")
EMBEDDING_API_KEY = os.environ.get("CSBREC_EMBEDDING_API_KEY", "")
if not EMBEDDING_URL or not EMBEDDING_API_KEY:
    raise RuntimeError(
        "Set CSBREC_EMBEDDING_URL and CSBREC_EMBEDDING_API_KEY before running this notebook."
    )


In [ ]:
def get_response(prompt):
    # Preserve the embedding model used by the upstream preprocessing pipeline.
    payload = {"model": "text-embedding-ada-002", "input": prompt}
    headers = {"Authorization": f"Bearer {EMBEDDING_API_KEY}"}
    for attempt in range(3):
        try:
            response = requests.post(
                EMBEDDING_URL, headers=headers, json=payload, timeout=60
            )
            response.raise_for_status()
            return response.json()["data"][0]["embedding"]
        except requests.RequestException as exc:
            status = exc.response.status_code if exc.response is not None else None
            # Retry transient failures only, and let the caller save its cache.
            if attempt == 2 or (status is not None and status != 429 and status < 500):
                raise
            time.sleep(2 ** attempt)


In [ ]:
cache_path = "./handled/item_emb.pkl"
if os.path.exists(cache_path):
    with open(cache_path, "rb") as cache_file:
        item_emb = pickle.load(cache_file)
else:
    item_emb = {}


In [ ]:
try:
    for key, value in tqdm(item_data.items()):
        if key not in item_emb:
            item_emb[key] = get_response(value)
finally:
    # Persist completed requests on success, failure or keyboard interruption.
    with open(cache_path, "wb") as cache_file:
        pickle.dump(item_emb, cache_file)


In [ ]:
len(item_emb)

In [ ]:
id_map = json.load(open("./handled/id_map.json", "r"))["id2item"]
emb_list = []
for id in range(1, len(item_emb)+1):
    meta_emb = item_emb[id_map[str(id)]]
    emb_list.append(meta_emb)

emb_list = np.array(emb_list)
pickle.dump(emb_list, open("./handled/itm_emb_np.pkl", "wb"))